In [0]:
# If this fails with 403, your Service Account lacks permissions
display(dbutils.fs.ls("gs://nawaf-etl-lake-dev-useast/raw"))

In [0]:
# If this hangs or fails, your GCP Project might have an API quota issue
spark.read.text("gs://nawaf-etl-lake-dev-useast/raw/crm/*.csv").limit(5).show()

In [0]:
# Test 1: Can we see the files?
try:
    files = dbutils.fs.ls("gs://nawaf-etl-lake-dev-useast/raw/crm")
    display(files)
    print("✅ GCS Connection Successful: Files are visible.")
except Exception as e:
    print(f"❌ GCS Connection Failed: {e}")

In [0]:
# Test 2: Can we access the 'nawaf' catalog?
try:
    spark.sql("CREATE CATALOG IF NOT EXISTS nawaf")
    spark.sql("CREATE SCHEMA IF NOT EXISTS nawaf.bronze_new")
    spark.sql("USE CATALOG nawaf")
    spark.sql("SHOW TABLES IN bronze_new")
    print("✅ Unity Catalog Access Successful: 'nawaf.bronze_new' is ready.")
except Exception as e:
    print(f"❌ Unity Catalog Access Failed: {e}")

In [0]:
# Test 3: Dry run of a single table ingestion
from pyspark.sql.functions import current_timestamp, col

try:
    # 1. Read a sample file
    sample_path = "gs://nawaf-etl-lake-dev-useast/raw/crm/cust_info.csv" # Update with an actual filename if needed
    df_test = spark.read.format("csv").option("header", "true").load(sample_path)
    
    # 2. Add metadata
    df_test = df_test.withColumn("_ingest_timestamp", current_timestamp())
    
    # 3. Attempt to write as a MANAGED table (no path specified)
    # This avoids the Overlap error by letting UC manage the storage
    df_test.write.format("delta").mode("overwrite").saveAsTable("nawaf.bronze_new.test_connection")
    
    print("✅ Write Successful: Data landed in Unity Catalog.")
    display(spark.table("nawaf.bronze_new.test_connection").limit(5))
except Exception as e:
    print(f"❌ Write Failed: {e}")

In [0]:
# 1. Define a dedicated path for catalog storage (NOT your raw folder)
# This folder will store the Delta tables that Unity Catalog manages for you.
CATALOG_STORAGE_PATH = "gs://nawaf-etl-lake-dev-useast/unity-catalog-storage/nawaf_catalog/"

try:
    # 2. Create the catalog with an explicit MANAGED LOCATION
    spark.sql(f"CREATE CATALOG IF NOT EXISTS nawaf MANAGED LOCATION '{CATALOG_STORAGE_PATH}'")
    
    # 3. Create the schema
    spark.sql("CREATE SCHEMA IF NOT EXISTS nawaf.bronze_new")
    
    # 4. Set context
    spark.sql("USE CATALOG nawaf")
    print("✅ Unity Catalog Access Successful: 'nawaf.bronze_new' is now linked to GCS.")
    
except Exception as e:
    print(f"❌ Still failing: {e}")
    print("\nPRO TIP: Ensure your Databricks 'External Location' in Admin settings includes the path above.")

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Configuration from previous steps
CATALOG = "nawaf"
SCHEMA = "bronze_new"
SAMPLE_SOURCE = "gs://nawaf-etl-lake-dev-useast/raw/crm/20260221135014660445_cust_info.csv"

try:
    print(f"--- Starting Test 3: Managed Write to {CATALOG}.{SCHEMA} ---")
    
    # 1. Read a single file from GCS
    # We use 'inferSchema' to see if Spark can parse the data types correctly
    df_test = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(SAMPLE_SOURCE)
    )
    
    # 2. Add a simple transformation
    df_test = df_test.withColumn("_test_ingest_time", current_timestamp())
    
    # 3. Write as a MANAGED table
    # IMPORTANT: Do NOT provide a 'path' here. 
    # UC will put this in your 'gs://.../unity-catalog-storage/nawaf_catalog/' folder automatically.
    table_name = f"{CATALOG}.{SCHEMA}.test_ingest_success"
    
    (df_test.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    
    print(f"✅ SUCCESS: Table created at {table_name}")
    
    # 4. Verify the data can be read back
    display(spark.table(table_name).limit(5))
    
except Exception as e:
    print(f"❌ Test 3 Failed: {e}")

In [0]:
# Run this to see the exact filenames
files = dbutils.fs.ls("gs://nawaf-etl-lake-dev-useast/raw/crm/")
for file in files:
    print(file.name)

In [0]:
from pyspark.sql.functions import current_timestamp, col

CATALOG = "nawaf"
SCHEMA = "bronze_new"
RAW_PATH = "gs://nawaf-etl-lake-dev-useast/raw/crm/"

try:
    # 1. Automatically find the first CSV file in the directory
    files = dbutils.fs.ls(RAW_PATH)
    csv_files = [f.path for f in files if f.name.endswith('.csv')]
    
    if not csv_files:
        raise Exception(f"No CSV files found in {RAW_PATH}")
    
    actual_file_path = csv_files[0]
    print(f"Reading file: {actual_file_path}")

    # 2. Read the file
    df_test = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(actual_file_path)
    )
    
    # 3. Add metadata
    df_test = df_test.withColumn("_test_ingest_time", current_timestamp())
    
    # 4. Write to Unity Catalog
    table_name = f"{CATALOG}.{SCHEMA}.test_ingest_success"
    (df_test.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    
    print(f"✅ SUCCESS: Data landed in {table_name}")
    display(spark.table(table_name).limit(5))

except Exception as e:
    print(f"❌ Test 3 still failing: {e}")
    

In [0]:
%sql
-- This forces Unity Catalog to unbind the path from the old 'workspace' catalog
-- Run these one by one to clear the 'Unauthorized' locks
DROP TABLE IF EXISTS workspace.bronze.cust_info;
DROP TABLE IF EXISTS workspace.bronze.prd_info;
-- If you have a volume, drop it too
DROP VOLUME IF EXISTS workspace.bronze.raw_files;

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Using your verified paths and catalog
RAW_PATH = "gs://nawaf-etl-lake-dev-useast/raw/crm/"
TARGET_TABLE = "nawaf.bronze_new.final_test_ingestion"

try:
    # 1. Read from the CRM directory (confirms Read/List permissions)
    df = spark.read.format("csv").option("header", "true").load(RAW_PATH)
    
    # 2. Add metadata for the Bronze layer
    df_bronze = df.withColumn("_ingest_time", current_timestamp())
    
    # 3. Write to the new catalog (confirms Managed Location permissions)
    # This avoids the 'Location Overlap' error by using Managed storage
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
    
    print(f"✅ Setup Complete! Data successfully landed in {TARGET_TABLE}")
    display(spark.table(TARGET_TABLE).limit(5))
    
except Exception as e:
    print(f"❌ Final validation failed: {e}")

In [0]:
%sql
-- This removes the metadata "lock" on your GCS files
DROP TABLE IF EXISTS workspace.bronze.cust_info;
DROP TABLE IF EXISTS workspace.bronze.prd_info;

-- If the error mentioned other tables, drop those as well
-- Example: DROP TABLE IF EXISTS workspace.bronze.sales_details;

In [0]:
%sql
-- Replace 'workspace.bronze' with the exact conflicting schema from your error message
DROP SCHEMA IF EXISTS workspace.bronze CASCADE;

In [0]:
from pyspark.sql.functions import current_timestamp

# Final check: Read from GCS and land in 'nawaf'
try:
    df = spark.read.format("csv").option("header", "true").load("gs://nawaf-etl-lake-dev-useast/raw/crm/")
    
    # Save as a MANAGED table in your new catalog
    df.withColumn("_ingest_time", current_timestamp()) \
      .write.format("delta").mode("overwrite") \
      .saveAsTable("nawaf.bronze_new.crm_cust_info")
    
    print("✅ SUCCESS: The pipeline is finally unlocked!")
    display(spark.table("nawaf.bronze_new.crm_cust_info").limit(5))
except Exception as e:
    print(f"❌ Still blocked: {e}")

In [0]:
%sql
-- This lists every table/volume UC has registered for that bucket
DESCRIBE EXTERNAL LOCATION `gs://nawaf-etl-lake-dev-useast/`;

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `gcs-test`;

In [0]:
%sql
-- Switch context to the catalog where the overlap exists
USE CATALOG workspace;

-- Drop the specific tables mentioned in the error message
DROP TABLE IF EXISTS bronze.cust_info;
DROP TABLE IF EXISTS bronze.prd_info;

-- If you have a volume blocking the path, drop it too
DROP VOLUME IF EXISTS bronze.raw_files;

-- Best practice: Drop the schema with CASCADE to clear all hidden locks
DROP SCHEMA IF EXISTS bronze CASCADE;

In [0]:
# Final validation test
try:
    path = "gs://nawaf-etl-lake-dev-useast/raw/crm/"
    df = spark.read.format("csv").option("header", "true").load(path)
    
    print(f"✅ SUCCESS: Path is finally unlocked!")
    display(df.limit(5))
    
    # Land the data in your new catalog
    df.write.format("delta").mode("overwrite").saveAsTable("nawaf.bronze_new.crm_cust_info")
    print("✅ SUCCESS: Data landed in nawaf.bronze_new.crm_cust_info")
    
except Exception as e:
    print(f"❌ Still blocked: {e}")

In [0]:
# Test: Can we finally READ from the raw path?
try:
    path = "gs://nawaf-etl-lake-dev-useast/raw/crm/"
    df_check = spark.read.format("csv").option("header", "true").load(path)
    print(f"✅ SUCCESS: The path is officially unlocked! Found {df_check.count()} rows.")
    display(df_check.limit(5))
except Exception as e:
    print(f"❌ Error during read: {e}")

In [0]:
# Test: Can we finally READ from the raw path?
try:
    path = "gs://nawaf-etl-lake-dev-useast/"
    df_check = spark.read.format("csv").option("header", "true").load(path)
    print(f"✅ SUCCESS: The path is officially unlocked! Found {df_check.count()} rows.")
    display(df_check.limit(5))
except Exception as e:
    print(f"❌ Error during read: {e}")

In [0]:
# Test 6: Ingesting from the RENAMED landing zone
from pyspark.sql.functions import current_timestamp

# Note the updated path name
NEW_RAW_PATH = "gs://nawaf-etl-lake-dev-useast/raw_new_nawaf/crm/"

try:
    # 1. Read from the new path
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .load(NEW_RAW_PATH)
    )
    
    # 2. Write as a MANAGED table to your 'nawaf' catalog
    # Managed tables are stored in your catalog's managed location,
    # which avoids future location overlap conflicts
    (df.withColumn("_ingest_time", current_timestamp())
       .write.format("delta")
       .mode("overwrite")
       .saveAsTable("nawaf.bronze_new.crm_cust_info"))
    
    print("✅ SUCCESS: The path is officially unlocked and data is landing in 'nawaf'!")
    display(spark.table("nawaf.bronze_new.crm_cust_info").limit(5))
except Exception as e:
    print(f"❌ Still blocked: {e}")

In [0]:
# Configuration
CATALOG = "nawaf"
SCHEMA = "bronze_new"

print(f"--- Cleaning all tables from {CATALOG}.{SCHEMA} ---")

# 1. Get the list of all tables in the schema
tables = spark.catalog.listTables(f"{CATALOG}.{SCHEMA}")

if not tables:
    print("No tables found to drop.")
else:
    for table in tables:
        full_table_name = f"{CATALOG}.{SCHEMA}.{table.name}"
        
        # 2. Execute the drop command
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
            print(f"Successfully dropped: {full_table_name}")
        except Exception as e:
            print(f"Failed to drop {full_table_name}: {e}")

print("\nCleanup complete. You now have a clean Bronze layer!")